# Exploring Spectrogram

In [1]:
import torch
import torch.nn as nn
import torchaudio
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import os
from typing import List

In [2]:
AUDIO_DIR = "../../data_processing/separate_scripts/golden_audio"
LABEL_DIR = "../../data_processing/separate_scripts/golden_breaks"
INDEX_SET = [0, 6, 16, 19]
interval_width = 20  # ms

In [3]:
class BreakDataset(Dataset):
    def __init__(self, indices: List[int] = INDEX_SET, audio_dir: str = AUDIO_DIR, label_dir: str = LABEL_DIR):
        self.indices = indices
        self.audio_dir = audio_dir
        self.label_dir = label_dir

        self.data = []
        for idx in indices:
            # Load audio and convert to spectrogram
            audio_path = os.path.join(audio_dir, f"{idx}.mp3")
            waveform, sr = torchaudio.load(audio_path)

            # Ensure mono channel - explicitly convert to single channel
            waveform = waveform.mean(dim=0, keepdim=True)

            # Compute mel spectrogram with adjusted parameters
            mel_spec = torchaudio.transforms.MelSpectrogram(
                sample_rate=sr,
                hop_length=int(sr * interval_width / 1000),
                n_mels=32
            )(waveform)

            # here, the mel_spec has the shape [1, n_mels=128, time_frames]
            # reshape it to [n_mels=128, time_frames]
            mel_spec = mel_spec.squeeze(0)

            # Load labels
            label_path = os.path.join(label_dir, f"{idx}.csv")
            df = pd.read_csv(label_path)
            labels = torch.tensor(df['break'].values, dtype=torch.bool)

            # Truncate to last break + 2
            last_break = labels.nonzero(as_tuple=True)[0][-1]
            mel_spec = mel_spec[:, :last_break+2]
            labels = labels[:last_break+2]
            
            print(f"Index: {idx}, Mel shape: {mel_spec.shape}, Labels shape: {labels.shape}")

            self.data.append((mel_spec, labels))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        return self.data[idx]

In [4]:
dataset = BreakDataset()
dataloader = DataLoader(
    dataset,
    batch_size=1,
    shuffle=True,
)


Index: 0, Mel shape: torch.Size([32, 11737]), Labels shape: torch.Size([11737])
Index: 6, Mel shape: torch.Size([32, 4970]), Labels shape: torch.Size([4970])
Index: 16, Mel shape: torch.Size([32, 9842]), Labels shape: torch.Size([9842])
Index: 19, Mel shape: torch.Size([32, 3642]), Labels shape: torch.Size([3642])


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

NUM_CLASSES  = 2  # because we have binary labels, duh

class CNNRNN(nn.Module):
    def __init__(self, n_mels=32, hidden_size=128, num_layers=1):
        super(CNNRNN, self).__init__()
        cnn_output_size = 16
        self.conv = nn.Conv1d(n_mels, cnn_output_size, kernel_size=3, padding=1)
        self.rnn = nn.LSTM(input_size=cnn_output_size, hidden_size=hidden_size, num_layers=num_layers, batch_first=True)
        # small sequence of linear
        self.fc = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Linear(64, NUM_CLASSES)
        )

    def forward(self, x, mask):
        # x: (batch, n_mels, time_frames)
        x = F.relu(self.conv(x))  # (batch, cnn_output_size, time_frames)
        x = x.permute(0, 2, 1)  # (batch, time_frames, cnn_output_size)
        x, _ = self.rnn(x)  # (batch, time_frames, hidden_size)
        x = self.fc(x)  # (batch, time_frames, num_classes)
        x = x * mask.unsqueeze(-1)
        return x

# Initialize the model
model = CNNRNN()


In [88]:
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Initialize the dataset and data loader
dataset = BreakDataset(INDEX_SET, AUDIO_DIR, LABEL_DIR)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

# Initialize the model, optimizer, and loss function
model = CNNRNN()
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCEWithLogitsLoss()

# Train the model
epochs = 3
for epoch in range(epochs):
    for batch in dataloader:
        mel_spec, labels = batch
        max_length = mel_spec.shape[1]
        mel_specs = []
        labels_list = []
        for i in range(mel_spec.shape[0]):
            mel_specs.append(F.pad(mel_spec[i], (0, max_length - mel_spec[i].shape[1])))
            labels_list.append(F.pad(labels[i], (0, max_length - labels[i].shape[0])))
        mel_specs = torch.stack(mel_specs)
        labels = torch.stack(labels_list)
        mask = (mel_specs!= 0).any(dim=1).float()
        output = model(mel_specs, mask)
        loss = criterion(output.view(-1, 2)[:, 1], labels.view(-1).float())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item()}")

    # Save the model
    torch.save(model.state_dict(), f"saved_models/model_epoch_{epoch+1}.pth")


Index: 0, Mel shape: torch.Size([32, 11737]), Labels shape: torch.Size([11737])
Index: 6, Mel shape: torch.Size([32, 4970]), Labels shape: torch.Size([4970])
Index: 16, Mel shape: torch.Size([32, 9842]), Labels shape: torch.Size([9842])
Index: 19, Mel shape: torch.Size([32, 3642]), Labels shape: torch.Size([3642])
Epoch 1, Loss: 0.6931471824645996
Epoch 2, Loss: 0.6931471824645996
Epoch 3, Loss: 0.6931471824645996


In [8]:
import torch
import torch.nn as nn
import torchaudio
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import os
from typing import List
import torch.nn.functional as F
import torch.optim as optim

AUDIO_DIR = "../../data_processing/separate_scripts/golden_audio"
LABEL_DIR = "../../data_processing/separate_scripts/golden_breaks"
INDEX_SET = [0, 6, 16, 19]
interval_width = 20  # ms
NUM_CLASSES = 2


class BreakDataset(Dataset):
    def __init__(self, indices: List[int] = INDEX_SET, audio_dir: str = AUDIO_DIR, label_dir: str = LABEL_DIR):
        self.indices = indices
        self.audio_dir = audio_dir
        self.label_dir = label_dir

        self.data = []
        for idx in indices:
            # Load audio and convert to spectrogram
            audio_path = os.path.join(audio_dir, f"{idx}.mp3")
            waveform, sr = torchaudio.load(audio_path)

            # Ensure mono channel
            waveform = waveform.mean(dim=0, keepdim=True)

            # Compute mel spectrogram
            mel_spec = torchaudio.transforms.MelSpectrogram(
                sample_rate=sr,
                hop_length=int(sr * interval_width / 1000),
                n_mels=32
            )(waveform)

            mel_spec = mel_spec.squeeze(0)

            # Add normalization
            mel_spec = torch.log(mel_spec + 1e-9)  # Log scale
            mel_spec = (mel_spec - mel_spec.mean()) / (mel_spec.std() + 1e-9)  # Normalize

            # Load labels
            label_path = os.path.join(label_dir, f"{idx}.csv")
            df = pd.read_csv(label_path)
            labels = torch.tensor(df['break'].values, dtype=torch.bool)

            # Truncate to last break + 2
            last_break = labels.nonzero(as_tuple=True)[0][-1]
            mel_spec = mel_spec[:, :last_break + 2]
            labels = labels[:last_break + 2]

            print(f"Index: {idx}, Mel shape: {mel_spec.shape}, Labels shape: {labels.shape}")

            self.data.append((mel_spec, labels))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        return self.data[idx]


class CNNRNN(nn.Module):
    def __init__(self, n_mels=32, hidden_size=128, num_layers=2):
        super(CNNRNN, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(n_mels, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(64, 32, kernel_size=3, padding=1),
            nn.ReLU()
        )
        self.rnn = nn.LSTM(
            input_size=32,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_size * 2, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, NUM_CLASSES)
        )

    def forward(self, x, mask):
        x = self.conv(x)
        x = x.permute(0, 2, 1)
        x, _ = self.rnn(x)
        x = self.fc(x)
        x = x * mask.unsqueeze(-1)
        return x


# Split data into train and validation sets
total_indices = len(INDEX_SET)
train_size = int(0.75 * total_indices)
train_indices = INDEX_SET[:train_size]
val_indices = INDEX_SET[train_size:]

train_dataset = BreakDataset(train_indices, AUDIO_DIR, LABEL_DIR)
val_dataset = BreakDataset(val_indices, AUDIO_DIR, LABEL_DIR)

train_dataloader = DataLoader(train_dataset, batch_size=1, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=1, shuffle=False)

# Initialize the model, optimizer, and loss function
model = CNNRNN()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=2)

# Calculate class weights for the loss function
all_labels = torch.cat([labels for _, labels in train_dataset])
pos_weight = torch.tensor([(all_labels == 0).sum() / (all_labels == 1).sum()])
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Training loop with validation and early stopping
epochs = 30  # Increased epochs since we have early stopping
best_loss = float('inf')
patience = 10
patience_counter = 0

for epoch in range(epochs):
    # Training phase
    model.train()
    train_loss = 0
    train_batches = 0

    for batch in train_dataloader:
        mel_spec, labels = batch
        max_length = mel_spec.shape[2]
        mel_specs = []
        labels_list = []

        for i in range(mel_spec.shape[0]):
            mel_specs.append(F.pad(mel_spec[i], (0, max_length - mel_spec[i].shape[1])))
            labels_list.append(F.pad(labels[i], (0, max_length - labels[i].shape[0])))

        mel_specs = torch.stack(mel_specs)
        labels = torch.stack(labels_list)
        mask = (mel_specs != 0).any(dim=1).float()

        optimizer.zero_grad()
        output = model(mel_specs, mask)
        loss = criterion(output.view(-1, 2)[:, 1], labels.view(-1).float())
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_batches += 1

    avg_train_loss = train_loss / train_batches

    # Validation phase
    model.eval()
    val_loss = 0
    val_batches = 0

    with torch.no_grad():
        for batch in val_dataloader:
            mel_spec, labels = batch
            max_length = mel_spec.shape[2]
            mel_specs = []
            labels_list = []

            for i in range(mel_spec.shape[0]):
                mel_specs.append(F.pad(mel_spec[i], (0, max_length - mel_spec[i].shape[1])))
                labels_list.append(F.pad(labels[i], (0, max_length - labels[i].shape[0])))

            mel_specs = torch.stack(mel_specs)
            labels = torch.stack(labels_list)
            mask = (mel_specs != 0).any(dim=1).float()

            output = model(mel_specs, mask)
            loss = criterion(output.view(-1, 2)[:, 1], labels.view(-1).float())

            val_loss += loss.item()
            val_batches += 1

    avg_val_loss = val_loss / val_batches

    # Learning rate scheduling
    scheduler.step(avg_val_loss)

    print(f"Epoch {epoch + 1}")
    print(f"Training Loss: {avg_train_loss:.4f}")
    print(f"Validation Loss: {avg_val_loss:.4f}")
    print(f"Learning Rate: {optimizer.param_groups[0]['lr']:.6f}")

    # Early stopping check
    if avg_val_loss < best_loss:
        best_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), 'best_model.pth')
        print("Saved new best model!")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping triggered!")
            break

    print("-" * 50)

print("Training completed!")


Index: 0, Mel shape: torch.Size([32, 11737]), Labels shape: torch.Size([11737])
Index: 6, Mel shape: torch.Size([32, 4970]), Labels shape: torch.Size([4970])
Index: 16, Mel shape: torch.Size([32, 9842]), Labels shape: torch.Size([9842])
Index: 19, Mel shape: torch.Size([32, 3642]), Labels shape: torch.Size([3642])
Epoch 1
Training Loss: 1.3343
Validation Loss: 1.7154
Learning Rate: 0.001000
Saved new best model!
--------------------------------------------------
Epoch 2
Training Loss: 1.3311
Validation Loss: 1.7081
Learning Rate: 0.001000
Saved new best model!
--------------------------------------------------
Epoch 3
Training Loss: 1.3278
Validation Loss: 1.6826
Learning Rate: 0.001000
Saved new best model!
--------------------------------------------------
Epoch 4
Training Loss: 1.3172
Validation Loss: 1.6387
Learning Rate: 0.001000
Saved new best model!
--------------------------------------------------
Epoch 5
Training Loss: 1.2877
Validation Loss: 1.5237
Learning Rate: 0.001000
Sa

# Training but better

In [9]:
import torch
import torch.nn as nn
import torchaudio
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import os
from typing import List
import torch.nn.functional as F
import torch.optim as optim

AUDIO_DIR = "../../data_processing/separate_scripts/golden_audio"
LABEL_DIR = "../../data_processing/separate_scripts/golden_breaks"
INDEX_SET = [0, 6, 19]  # 16
interval_width = 20  # ms
NUM_CLASSES = 2


class BreakDataset(Dataset):
    def __init__(self, indices: List[int] = INDEX_SET, audio_dir: str = AUDIO_DIR, label_dir: str = LABEL_DIR):
        self.indices = indices
        self.audio_dir = audio_dir
        self.label_dir = label_dir

        self.data = []
        for idx in indices:
            # Load audio and convert to spectrogram
            audio_path = os.path.join(audio_dir, f"{idx}.mp3")
            waveform, sr = torchaudio.load(audio_path)

            # Ensure mono channel
            waveform = waveform.mean(dim=0, keepdim=True)

            # Compute mel spectrogram
            mel_spec = torchaudio.transforms.MelSpectrogram(
                sample_rate=sr,
                hop_length=int(sr * interval_width / 1000),
                n_mels=32
            )(waveform)

            mel_spec = mel_spec.squeeze(0)

            # Add normalization
            mel_spec = torch.log(mel_spec + 1e-9)  # Log scale
            mel_spec = (mel_spec - mel_spec.mean()) / (mel_spec.std() + 1e-9)  # Normalize

            # Load labels
            label_path = os.path.join(label_dir, f"{idx}.csv")
            df = pd.read_csv(label_path)
            labels = torch.tensor(df['break'].values, dtype=torch.bool)

            # Truncate to last break + 2
            last_break = labels.nonzero(as_tuple=True)[0][-1]
            mel_spec = mel_spec[:, :last_break + 2]
            labels = labels[:last_break + 2]

            print(f"Index: {idx}, Mel shape: {mel_spec.shape}, Labels shape: {labels.shape}")

            self.data.append((mel_spec, labels))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        return self.data[idx]


class CNNRNN(nn.Module):
    def __init__(self, n_mels=32, hidden_size=128, num_layers=2):
        super(CNNRNN, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(n_mels, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(64, 32, kernel_size=3, padding=1),
            nn.ReLU()
        )
        self.rnn = nn.LSTM(
            input_size=32,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_size * 2, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, NUM_CLASSES)
        )

    def forward(self, x, mask):
        x = self.conv(x)
        x = x.permute(0, 2, 1)
        x, _ = self.rnn(x)
        x = self.fc(x)
        x = x * mask.unsqueeze(-1)
        return x


# Split data into train and validation sets
total_indices = len(INDEX_SET)
train_size = int(0.75 * total_indices)
train_indices = INDEX_SET[:train_size]
val_indices = INDEX_SET[train_size:]

train_dataset = BreakDataset(train_indices, AUDIO_DIR, LABEL_DIR)
val_dataset = BreakDataset(val_indices, AUDIO_DIR, LABEL_DIR)

train_dataloader = DataLoader(train_dataset, batch_size=1, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=1, shuffle=False)

# Initialize the model, optimizer, and loss function
model = CNNRNN()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=2)

# Calculate class weights for the loss function
all_labels = torch.cat([labels for _, labels in train_dataset])
pos_weight = torch.tensor([(all_labels == 0).sum() / (all_labels == 1).sum()])
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# Training loop with validation and early stopping
epochs = 30  # Increased epochs since we have early stopping
best_loss = float('inf')
patience = 10
patience_counter = 0

for epoch in range(epochs):
    # Training phase
    model.train()
    train_loss = 0
    train_batches = 0

    for batch in train_dataloader:
        mel_spec, labels = batch
        max_length = mel_spec.shape[2]
        mel_specs = []
        labels_list = []

        for i in range(mel_spec.shape[0]):
            mel_specs.append(F.pad(mel_spec[i], (0, max_length - mel_spec[i].shape[1])))
            labels_list.append(F.pad(labels[i], (0, max_length - labels[i].shape[0])))

        mel_specs = torch.stack(mel_specs)
        labels = torch.stack(labels_list)
        mask = (mel_specs != 0).any(dim=1).float()

        optimizer.zero_grad()
        output = model(mel_specs, mask)
        loss = criterion(output.view(-1, 2)[:, 1], labels.view(-1).float())
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_batches += 1

    avg_train_loss = train_loss / train_batches

    # Validation phase
    model.eval()
    val_loss = 0
    val_batches = 0

    with torch.no_grad():
        for batch in val_dataloader:
            mel_spec, labels = batch
            max_length = mel_spec.shape[2]
            mel_specs = []
            labels_list = []

            for i in range(mel_spec.shape[0]):
                mel_specs.append(F.pad(mel_spec[i], (0, max_length - mel_spec[i].shape[1])))
                labels_list.append(F.pad(labels[i], (0, max_length - labels[i].shape[0])))

            mel_specs = torch.stack(mel_specs)
            labels = torch.stack(labels_list)
            mask = (mel_specs != 0).any(dim=1).float()

            output = model(mel_specs, mask)
            loss = criterion(output.view(-1, 2)[:, 1], labels.view(-1).float())

            val_loss += loss.item()
            val_batches += 1

    avg_val_loss = val_loss / val_batches

    # Learning rate scheduling
    scheduler.step(avg_val_loss)

    print(f"Epoch {epoch + 1}")
    print(f"Training Loss: {avg_train_loss:.4f}")
    print(f"Validation Loss: {avg_val_loss:.4f}")
    print(f"Learning Rate: {optimizer.param_groups[0]['lr']:.6f}")

    # Early stopping check
    if avg_val_loss < best_loss:
        best_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), 'best_model.pth')
        print("Saved new best model!")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping triggered!")
            break

    print("-" * 50)

print("Training completed!")


Index: 0, Mel shape: torch.Size([32, 11737]), Labels shape: torch.Size([11737])
Index: 6, Mel shape: torch.Size([32, 4970]), Labels shape: torch.Size([4970])
Index: 19, Mel shape: torch.Size([32, 3642]), Labels shape: torch.Size([3642])
Epoch 1
Training Loss: 1.3261
Validation Loss: 1.6323
Learning Rate: 0.001000
Saved new best model!
--------------------------------------------------
Epoch 2
Training Loss: 1.3196
Validation Loss: 1.6089
Learning Rate: 0.001000
Saved new best model!
--------------------------------------------------
Epoch 3
Training Loss: 1.3111
Validation Loss: 1.5654
Learning Rate: 0.001000
Saved new best model!
--------------------------------------------------
Epoch 4
Training Loss: 1.2963
Validation Loss: 1.4892
Learning Rate: 0.001000
Saved new best model!
--------------------------------------------------
Epoch 5
Training Loss: 1.2774
Validation Loss: 1.4126
Learning Rate: 0.001000
Saved new best model!
--------------------------------------------------
Epoch 6


# Inference

In [12]:
import torch
import torchaudio
import pandas as pd
import os
import torch.nn.functional as F

# Reuse the model definition from training script
class CNNRNN(nn.Module):
    def __init__(self, n_mels=32, hidden_size=128, num_layers=2):
        super(CNNRNN, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(n_mels, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(64, 32, kernel_size=3, padding=1),
            nn.ReLU()
        )
        self.rnn = nn.LSTM(
            input_size=32,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_size * 2, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, NUM_CLASSES)
        )

    def forward(self, x, mask):
        x = self.conv(x)
        x = x.permute(0, 2, 1)
        x, _ = self.rnn(x)
        x = self.fc(x)
        x = x * mask.unsqueeze(-1)
        return x

def prepare_input(audio_path, interval_width=20):
    # Load audio
    waveform, sr = torchaudio.load(audio_path)

    # Ensure mono channel
    waveform = waveform.mean(dim=0, keepdim=True)

    # Compute mel spectrogram
    mel_spec_transform = torchaudio.transforms.MelSpectrogram(
        sample_rate=sr,
        hop_length=int(sr * interval_width / 1000),
        n_mels=32
    )
    mel_spec = mel_spec_transform(waveform)
    mel_spec = mel_spec.squeeze(0)

    # Normalization (same as in training)
    mel_spec = torch.log(mel_spec + 1e-9)
    mel_spec = (mel_spec - mel_spec.mean()) / (mel_spec.std() + 1e-9)

    return mel_spec, sr

def inference(model_path, audio_path, output_csv_path):
    # Load the trained model
    model = CNNRNN()
    model.load_state_dict(torch.load(model_path, weights_only=True))
    model.eval()

    # Prepare input
    mel_spec, sr = prepare_input(audio_path)

    # Prepare batch dimensions
    mel_spec = mel_spec.unsqueeze(0)  # Add batch dimension

    # Create mask
    mask = (mel_spec != 0).any(dim=1).float()

    # Inference
    with torch.no_grad():
        outputs = model(mel_spec, mask)

    # Convert outputs to probabilities
    probabilities = torch.sigmoid(outputs.squeeze())

    # Convert to numpy for easier handling
    probs_np = probabilities.numpy()

    # Create DataFrame
    df = pd.DataFrame({
        'time_ms': [i * interval_width for i in range(len(probs_np))],
        'break_probability': probs_np[:, 1],  # Probability of being a break
        'is_break': (probs_np[:, 1] > 0.5).astype(int)  # Threshold at 0.5
    })

    # Save to CSV
    df.to_csv(output_csv_path, index=False)

    print(f"Inference completed. Results saved to {output_csv_path}")
    return df

# Example usage
if __name__ == "__main__":
    # Paths
    MODEL_PATH = 'best_model.pth'
    AUDIO_PATH = '../../data_processing/separate_scripts/golden_audio/16.mp3'
    OUTPUT_CSV_PATH = 'break_predictions.csv'

    # Run inference
    results = inference(MODEL_PATH, AUDIO_PATH, OUTPUT_CSV_PATH)

    # Optional: print some statistics
    print(f"Total predictions: {len(results)}")
    print(f"Number of predicted breaks: {results['is_break'].sum()}")


Inference completed. Results saved to break_predictions.csv
Total predictions: 11038
Number of predicted breaks: 8061


In [18]:
# for an entire directory of audio (all following the name format of {INTEGER}.mp3), run inference on all of them and put their corresponding csvs in a directory
input_dir = '../../data/clean/audio/vocals'
output_dir = 'model_1_predicted_segment_breaks'

def inference_dir(model_path, input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    for audio_file in os.listdir(input_dir):
        if not audio_file.endswith('.mp3'):
            continue
        audio_path = os.path.join(input_dir, audio_file)
        output_csv_path = os.path.join(output_dir, f"{audio_file[:-4]}_breaks.csv")
        inference(model_path, audio_path, output_csv_path)
        
# Run inference on the directory
inference_dir('best_model.pth', input_dir, output_dir)

Inference completed. Results saved to model_1_predicted_segment_breaks/0_breaks.csv
Inference completed. Results saved to model_1_predicted_segment_breaks/10_breaks.csv
Inference completed. Results saved to model_1_predicted_segment_breaks/11_breaks.csv
Inference completed. Results saved to model_1_predicted_segment_breaks/12_breaks.csv
Inference completed. Results saved to model_1_predicted_segment_breaks/13_breaks.csv
Inference completed. Results saved to model_1_predicted_segment_breaks/14_breaks.csv
Inference completed. Results saved to model_1_predicted_segment_breaks/15_breaks.csv
Inference completed. Results saved to model_1_predicted_segment_breaks/16_breaks.csv
Inference completed. Results saved to model_1_predicted_segment_breaks/17_breaks.csv
Inference completed. Results saved to model_1_predicted_segment_breaks/18_breaks.csv
Inference completed. Results saved to model_1_predicted_segment_breaks/19_breaks.csv
Inference completed. Results saved to model_1_predicted_segment_br

In [22]:
import pandas as pd
import numpy as np
import os
import glob

def convert_probabilities_to_breaks(input_csv, index, output_dir='', probability_threshold=0.5, min_break_duration=100):
    # Read the input CSV
    df = pd.read_csv(input_csv)

    # Initialize lists to store break segments
    break_segments = []

    # Track current break state
    in_break = False
    break_start = None

    for i in range(len(df)):
        # Calculate start and end times
        start_time = 20 * i
        end_time = 20 * (i + 1)

        # Check if current time is a break
        is_break = df.loc[i, 'break_probability'] > probability_threshold

        # State machine for break detection
        if is_break and not in_break:
            # Start of a new break
            break_start = start_time
            in_break = True

        elif not is_break and in_break:
            # End of a break
            # Only record if break is longer than minimum duration
            if end_time - break_start >= min_break_duration:
                break_segments.append({
                    'start': break_start,
                    'end': end_time,
                    'break': 1
                })

            # Reset break tracking
            break_start = None
            in_break = False

    # Handle case where break continues to the end
    if in_break:
        if end_time - break_start >= min_break_duration:
            break_segments.append({
                'start': break_start,
                'end': end_time,
                'break': 1
            })

    # Convert to DataFrame
    breaks_df = pd.DataFrame(break_segments)

    # Create output filename
    output_csv = os.path.join(output_dir, f"{index}_breaks.csv")

    # Save to CSV
    breaks_df.to_csv(output_csv, index=False)

    print(f"Converted breaks saved to {output_csv}")
    print(f"Total breaks detected: {len(breaks_df)}")

    return breaks_df

def process_all_predictions(prediction_dir, output_dir):
    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)

    # Find all prediction CSV files
    prediction_files = glob.glob(os.path.join(prediction_dir, '*_predictions.csv'))

    # Extract indices from filenames
    indices = []
    for file_path in prediction_files:
        filename = os.path.basename(file_path)
        try:
            index = int(filename.split('_')[0])
            indices.append(index)
        except (ValueError, IndexError):
            print(f"Skipping file with non-standard name: {filename}")

    # Sort indices
    indices.sort()

    # Process each prediction file
    for index in indices:
        # Construct input CSV path
        input_csv = os.path.join(prediction_dir, f"{index}_breaks.csv")

        # Check if file exists
        if not os.path.exists(input_csv):
            print(f"Warning: No prediction file found for index {index}")
            continue

        # Convert breaks
        convert_probabilities_to_breaks(
            input_csv,
            index,
            output_dir,
            probability_threshold=0.5,
            min_break_duration=100
        )

# Main execution
if __name__ == "__main__":
    # Directories
    prediction_dir = 'model_1_predicted_segment_breaks/'  # Directory with prediction CSVs
    output_dir = 'breaks/'  # Directory to save break CSVs

    # Process all prediction files
    process_all_predictions(prediction_dir, output_dir)


In [23]:
import pandas as pd
import os
import glob

def collate_break_files(input_dir, output_file='segment_breaks.csv'):
    # Get all break CSV files
    break_files = glob.glob(os.path.join(input_dir, '*_breaks.csv'))

    # Initialize list to store all breaks
    all_breaks = []

    # Process each break file
    for file_path in sorted(break_files):
        # Extract INDEX from filename
        file_index = int(os.path.basename(file_path).split('_')[0])

        # Read the break file
        try:
            df = pd.read_csv(file_path)

            # Add file column
            df['file'] = file_index

            # Rename 'break' column to 'token' if it exists
            if 'break' in df.columns:
                df = df.rename(columns={'break': 'token'})

            # Append to list
            all_breaks.append(df)

        except Exception as e:
            print(f"Error processing {file_path}: {e}")
            continue

    # Concatenate all dataframes
    if all_breaks:
        combined_df = pd.concat(all_breaks, ignore_index=True)

        # Reorder columns
        combined_df = combined_df[['file', 'start', 'end', 'token']]

        # Add index column (starting from 0)
        combined_df.insert(0, 'index', range(len(combined_df)))

        # Save to CSV
        combined_df.to_csv(output_file, index=False)

        print(f"Successfully created {output_file}")
        print(f"Total segments: {len(combined_df)}")
        print(f"Files processed: {len(break_files)}")

        return combined_df
    else:
        print("No break files were processed successfully")
        return None

if __name__ == "__main__":
    # Example usage
    input_directory = "model_1_predicted_segment_breaks/"  # Directory containing INDEX_breaks.csv files
    output_file = "segment_breaks.csv"

    df = collate_break_files(input_directory, output_file)

    # Optional: Display first few rows
    if df is not None:
        print("\nFirst few rows of the combined CSV:")
        print(df.head())


KeyError: "['start', 'end', 'token'] not in index"